web scraping

In [1]:
!pip install requests
!pip install beautifulsoup4

In [ ]:
import requests

# fetch html
url = "https://quotes.toscrape.com"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url, headers=headers)

In [ ]:
from bs4 import BeautifulSoup

# parse html
soup = BeautifulSoup(response.text, 'html.parser')

In [5]:
# extract data
quotes = []
quote_boxes = soup.find_all(
    "div",
    class_="quote",
)

for box in quote_boxes:
    quote = {
        "author": box.find("small", class_="author").text.strip(),
        "lines": box.find("span", class_="text").text.strip(),
    }
    quotes.append(quote)


In [ ]:
# save to csv
import csv

filename = "quotes.csv"
with open(filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["author", "lines"])
    writer.writeheader()
    for q in quotes:
        writer.writerow(q)

In [ ]:
# checking
for q in quotes[:5]:
    print(q)

{'author': 'Albert Einstein', 'lines': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'}
{'author': 'J.K. Rowling', 'lines': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”'}
{'author': 'Albert Einstein', 'lines': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”'}
{'author': 'Jane Austen', 'lines': '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”'}
{'author': 'Marilyn Monroe', 'lines': "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”"}


tokenization and embedding

In [10]:
!pip install nltk sentence-transformers

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   -

In [11]:
!pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.8 MB 5.2 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.8 MB 13.3 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.8 MB 15.1 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 15.2 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]

In [12]:
import pandas as pd

df = pd.read_csv("quotes.csv")

In [ ]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

all_sentences = []

for line in df["lines"]:
    sentence = sent_tokenize(line)
    all_sentences.extend(sentence)

print(all_sentences)

['“The world as we have created it is a process of our thinking.', 'It cannot be changed without changing our thinking.”', '“It is our choices, Harry, that show what we truly are, far more than our abilities.”', '“There are only two ways to live your life.', 'One is as though nothing is a miracle.', 'The other is as though everything is a miracle.”', '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”', "“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”", '“Try not to become a man of success.', 'Rather become a man of value.”', '“It is better to be hated for what you are than to be loved for what you are not.”', '“I have not failed.', "I've just found 10,000 ways that won't work.”", "“A woman is like a tea bag; you never know how strong it is until it's in hot water.”", '“A day without sunshine is like, you know, night.”']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\VICTUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [21]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(all_sentences)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9517.81it/s]


In [22]:
embeddings.shape

(15, 384)

In [23]:
similarities = model.similarity(embeddings, embeddings)
print(similarities)

tensor([[1.0000, 0.4512, 0.3478, 0.3178, 0.1760, 0.2981, 0.2314, 0.3894, 0.3563,
         0.2800, 0.2833, 0.2832, 0.1202, 0.2203, 0.2658],
        [0.4512, 1.0000, 0.3916, 0.3403, 0.1477, 0.2483, 0.2686, 0.3032, 0.3035,
         0.3748, 0.3091, 0.1324, 0.2883, 0.1840, 0.2003],
        [0.3478, 0.3916, 1.0000, 0.3632, 0.1872, 0.3451, 0.4213, 0.3923, 0.3844,
         0.3550, 0.4838, 0.3618, 0.1394, 0.4079, 0.3028],
        [0.3178, 0.3403, 0.3632, 1.0000, 0.3462, 0.3793, 0.2250, 0.3012, 0.4610,
         0.3214, 0.4143, 0.2980, 0.3267, 0.3074, 0.3316],
        [0.1760, 0.1477, 0.1872, 0.3462, 1.0000, 0.7760, 0.1064, 0.3045, 0.1992,
         0.0532, 0.0915, 0.2152, 0.2070, 0.1536, 0.1686],
        [0.2981, 0.2483, 0.3451, 0.3793, 0.7760, 1.0000, 0.2147, 0.4150, 0.2414,
         0.1863, 0.2770, 0.2351, 0.1475, 0.2376, 0.2821],
        [0.2314, 0.2686, 0.4213, 0.2250, 0.1064, 0.2147, 1.0000, 0.5488, 0.4015,
         0.3611, 0.4227, 0.3102, 0.1031, 0.4579, 0.3584],
        [0.3894, 0.3032, 0.